## Data

#### Using RecursiveUrlLoader

##### Adding extractor: To parse HTML into a more human/LLM-friendly format

In [ ]:
import re
from bs4 import BeautifulSoup
from langchain.document_loaders.recursive_url_loader import RecursiveUrlLoader
from langchain.document_loaders import PyPDFLoader
from langchain.utils.html import PREFIXES_TO_IGNORE_REGEX, SUFFIXES_TO_IGNORE_REGEX

# This function is used to process and clean the HTML content fetched from each URL
def bs4_extractor(html: str) -> str:
    soup = BeautifulSoup(html, "html5lib")  # Parse the HTML string using an XML parser
    return re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Extract text and remove HTML tags + handle extra lines

# List of all URLs to process
urls = [
    "https://licensespring.com/blog/glossary/open-source-software/",
    "https://opensource.guide/maintaining-balance-for-open-source-maintainers/",
    "https://opensource.guide/how-to-contribute/",
    "https://opensource.guide/starting-a-project/",
    "https://opensource.guide/finding-users/",
    "https://opensource.guide/building-community/",
    "https://opensource.guide/best-practices/",
    "https://opensource.guide/leadership-and-governance/",
    "https://opensource.guide/getting-paid/",
    "https://opensource.guide/code-of-conduct/",
    "https://opensource.guide/metrics/",
    "https://opensource.guide/legal/"
]

# Initialize an empty list to store documents
all_documents = []

# Load web data
for url in urls:
    try:
        loader = RecursiveUrlLoader(
            url,
            max_depth=4,  # Limits the recursion depth to 4 levels
            prevent_outside=True,  # Prevents following links to external domains
            timeout=600,  # Specifies the maximum time to wait for a page response
            extractor=bs4_extractor,
            link_regex=(
                f"href=[\"']{PREFIXES_TO_IGNORE_REGEX}((?:{SUFFIXES_TO_IGNORE_REGEX}.)*?)"
                r"(?:[\#'\"]|\/[\#'\"])"  # Matches links with href attributes in HTML
            )
        )
        documents = loader.load()  # Load documents for the current URL
        all_documents.extend(documents)  # Add documents to the combined list
    except Exception as e:
        print(f"Failed to process URL {url}: {e}")

# Load PDF data
pdf_loader1 = PyPDFLoader(file_path="OPEN SOURCE SOFTWARE GUIDELINES.pdf")
pdf_loader2 = PyPDFLoader(file_path="Creating Impactful.pdf")
pdf_data1 = pdf_loader1.load()
pdf_data2 = pdf_loader2.load()

# Add PDF documents to all_documents
all_documents.extend(pdf_data1)
all_documents.extend(pdf_data2)

# Print the content of each document
for idx, doc in enumerate(all_documents):
    print(f"Document {idx + 1} from Source: {doc.metadata.get('source', 'Unknown Source')}")
    print(doc.page_content[:])  # Print the full content or a portion of it
    print("-" * 80)  # Separator for clarity


## Split documents

To handle lengthy text efficiently, the Langchain text splitter divides text into smaller, semantically meaningful units and combines them into larger chunks with defined size and overlap. Here, I used the RecursiveCharacterTextSplitter to process scraped documents into manageable chunks while preserving context continuity.

`chunk_size` and `chunk_overlap` effects to the prompt size

execeed promt size causes error `prompt size exceeds the context window size and cannot be processed`

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

#chunk size: maximum number of characters each chunk can contain
#chunk oberlap: the number of overlapping characters between consecutive chunks, ensure context continuity between chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200) #Splits each document into chunks
chunked_documents = text_splitter.split_documents(all_documents)
print(chunked_documents)
#To print the resulting chunks 
# for idx, chunk in enumerate(chunked_documents):
#     print(f"Chunk {idx + 1} (Source: {chunk.metadata.get('source', 'Unknown Source')}):")
#     print(chunk.page_content)
#     print("-" * 80)  # Separator for clarity

##  Create Vector Embedding

After splitting the text, it is converted into vector embeddings using machine learning models like HuggingFace's all-MiniLM-L6-v2. These high-dimensional vectors capture semantic meanings and enable efficient operations such as grouping, searching, and measuring sentence similarity based on semantic closeness, surpassing traditional keyword-based methods.

#### configuration

In [ ]:
import os

# Set environment variables directly in the notebook
os.environ['INIT_INDEX'] = 'true'
os.environ['INDEX_PERSIST_DIRECTORY'] = './data1/chromadb'
os.environ['TARGET_URL'] = 'https://open5gs.org/open5gs/docs/'
os.environ['HTTP_PORT'] = '7654'
os.environ['MONGO_HOST'] = 'localhost'
os.environ['MONGO_PORT'] = '27017'
os.environ['MONGO_USER'] = 'testuser'
os.environ['MONGO_PASS'] = 'testpass'


In [ ]:

# define init index
INIT_INDEX = os.getenv('INIT_INDEX', 'false').lower() == 'true'

# vector index persist directory
INDEX_PERSIST_DIRECTORY = os.getenv('INDEX_PERSIST_DIRECTORY', "./data1/chromadb")

# target url to scrape
TARGET_URL =  os.getenv('TARGET_URL', "https://open5gs.org/open5gs/docs/")

# http api port
HTTP_PORT = os.getenv('HTTP_PORT', 7654)

# mongodb config host, username, password
MONGO_HOST = os.getenv('MONGO_HOST', 'localhost')
MONGO_PORT = os.getenv('MONGO_PORT', 27017)
MONGO_USER = os.getenv('MONGO_USER', 'testuser')
MONGO_PASS = os.getenv('MONGO_PASS', 'testpass')

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# Initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


## Store Vector Embedding in Chroma

Chroma (ChromaDB) is an open-source vector database that stores embeddings and their metadata, enabling efficient semantic search by processing text-based data semantically, unlike traditional databases. It enhances the system's ability to quickly retrieve and compare relevant information, improving the accuracy of responses to user queries.

In [ ]:
# Helper function to split data into batches
def batch_data(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

# Split documents into smaller batches
BATCH_SIZE = 166  # Maximum allowed batch size
for batch in batch_data(chunked_documents, BATCH_SIZE):
    # Add each batch to the Chroma vector store
    vectordb = Chroma.from_documents(
        documents=batch,
        embedding=embeddings,
        persist_directory=INDEX_PERSIST_DIRECTORY
    )


In [ ]:
# Check if the vector store contains any documents
print(f"Number of documents in the vector store: {len(vectordb)}")

In [ ]:
from langchain_chroma import Chroma
# load index
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectordb = Chroma(persist_directory=INDEX_PERSIST_DIRECTORY,embedding_function=embeddings)

## User Ask Question

The system offers an API that allows users to ask questions related to Open5GS documentation, with user sessions identified by a user_id for tracking. The API is designed for ease of use, enabling intuitive interactions, and in real-world scenarios, user identification could be managed via an Authorization header.

## Create Vector Embedding of Question

When a user submits a question through the API, the system converts it into a vector embedding, which is automatically generated by the ConversationalRetrievalChain, enabling semantic search of relevant documents in the vector database.

In [ ]:
from langchain_community.llms import Ollama
from langchain.llms import OpenAI
# llama2 llm which runs with ollama
# ollama expose an api for the llam in `localhost:11434`
llm = Ollama(
    model="llama3.2",
    base_url="http://localhost:11434",
    verbose=True, #provide detailed logs, messages, or output about what it's doing
)

In [ ]:
from langchain.chains import ConversationalRetrievalChain

# ConversationalRetrievalChain!!! A changer


# Create conversation with the correct method (invoke instead of __call__)
conversation = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=vectordb.as_retriever(),  # Convert the vector database to a retriever
    return_source_documents=True,  
    verbose=False,  # Disables detailed logging
)

# Initialize chat history
chat_history = []


In [ ]:
# Ask the first question
question = "Give me the definition of OSS?"

response = conversation.invoke({"question": question, "chat_history": chat_history})

# Extract the answer from the response
answer = response['answer']

# Print the answer
print(answer)

# Append the question and answer to chat history for the next conversation turn
chat_history.append((question, answer))

In [ ]:
# Repeat the process for subsequent questions
question_2 = "What is early determination of distribution policy?"
response_2 = conversation.invoke({"question": question_2, "chat_history": chat_history})
answer_2 = response_2['answer']
print(answer_2)

# Append the second question and answer to chat history
chat_history.append((question_2, answer_2))

In [ ]:
# Repeat the process for subsequent questions
question_3 = "What are the key characteristics of Open-source software?"
response_3 = conversation.invoke({"question": question_3, "chat_history": chat_history})
answer_3 = response_3['answer']
print(answer_3)

# Append the second question and answer to chat history
chat_history.append((question_3, answer_3))

In [ ]:
print(llm) 

In [ ]:
#changer le modele à quen2.5:14b
#choix model de embedding
#quelle justifie technique de RAG: langchain 'prompts particuliers'
#utiliser autre parser
#utilisr d'autres methodes pour preporcessing
#decoupage en chunks: par ponctuation, titres...